# 7.1. Pipelines and composite estimators

pipeline: 顺序执行。除最后之外，其余都是`transformer`
比如：填充缺失->标准化->regressor

ColumnTransformer: 并联执行。对不同的列做，不影响互相.最后拼接
比如：numeric列做标准化，category列做onehot

FeatureUnion: 与ColumnTransformer对比，这个会对一列做多个特征，最后拼接
比如：对列同时保留 原始值和平方项


In [15]:
from sklearn.pipeline import Pipeline
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest

## pipeline

make_pipeline vs. Pipeline

- Pipeline is more flexible.make_pipeline is more simple
- Pipeline needs us manually define name.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([
    ('my_scaler', StandardScaler()),
    ('my_classifier', LogisticRegression())
])
print(pipe.named_steps['my_scaler'])   # 获取

StandardScaler()


In [9]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pipe = make_pipeline(
    StandardScaler(),
    LogisticRegression()
)
print(pipe.named_steps['standardscaler'])  # auto-generated name

StandardScaler()


### tracking features name

保证pipeline链上有这个方法

In [16]:
iris = load_iris()

In [18]:
pipe = Pipeline(steps=[
    ('select', SelectKBest(k=2)),
    ('clf', LogisticRegression())
])

In [20]:
pipe.fit(iris.data, iris.target)

,steps,"[('select', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,score_func,<function f_c...00197F6D401F0>
,k,2
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True


In [25]:
pipe[:-1].get_feature_names_out()

array(['x2', 'x3'], dtype=object)

## ColumnTransformer for heterogeneous data
不同类型的特征需要不同处理。

可以通过`make_column_selector`获取类型列

## TransformedTargetRegressor

针对回归模型

对于目标y而言，经常需要拟合前进行`log`等变换，训练完，预测又要`exp`逆回来。

这里把他包装起来

In [5]:
from sklearn.datasets import make_regression
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.compose import TransformedTargetRegressor

type

In [8]:
help(TransformedTargetRegressor)

Help on class TransformedTargetRegressor in module sklearn.compose._target:

class TransformedTargetRegressor(sklearn.base.RegressorMixin, sklearn.base.BaseEstimator)
 |  TransformedTargetRegressor(regressor=None, *, transformer=None, func=None, inverse_func=None, check_inverse=True)
 |  
 |  Meta-estimator to regress on a transformed target.
 |  
 |  Useful for applying a non-linear transformation to the target `y` in
 |  regression problems. This transformation can be given as a Transformer
 |  such as the :class:`~sklearn.preprocessing.QuantileTransformer` or as a
 |  function and its inverse such as `np.log` and `np.exp`.
 |  
 |  The computation during :meth:`fit` is::
 |  
 |      regressor.fit(X, func(y))
 |  
 |  or::
 |  
 |      regressor.fit(X, transformer.transform(y))
 |  
 |  The computation during :meth:`predict` is::
 |  
 |      inverse_func(regressor.predict(X))
 |  
 |  or::
 |  
 |      transformer.inverse_transform(regressor.predict(X))
 |  
 |  Read more in the :r

In [15]:
y.shape

(1000,)

### custom estimator
一般不用，而是`FunctionTransformer`

`estimator` 是最基本的对象，实现了`fit`方法
```python
estimator = estimator.fit(data, targets)
estimator = estimator.fit(data)
```

`predictor`对象用于监督学习
```py
predictor = predictor.predict(data)
predictor = predictor.predict_proba(data) # 分类算法
```

`transformer`对象用于修改数据
```python
newdata = transformer.transform(data)
newdata = transformer.fit_transform(data) # 
```

`model`对象表示模型，可以给出得分metrics等
```python
score = model.score(data)
```

官方提供了[模板](https://github.com/scikit-learn-contrib/project-template/blob/main/skltemplate/_template.py#L227)

In [ ]:
# Note that the mixin class should always be on the left of `BaseEstimator` to ensure
# the MRO works as expected.
class TemplateTransformer(TransformerMixin, BaseEstimator):
    """An example transformer that returns the element-wise square root.

    For more information regarding how to build your own transformer, read more
    in the :ref:`User Guide <user_guide>`.

    Parameters
    ----------
    demo_param : str, default='demo'
        A parameter used for demonstation of how to pass and store paramters.

    Attributes
    ----------
    n_features_in_ : int
        Number of features seen during :term:`fit`.

    feature_names_in_ : ndarray of shape (`n_features_in_`,)
        Names of features seen during :term:`fit`. Defined only when `X`
        has feature names that are all strings.
    """

    # This is a dictionary allowing to define the type of parameters.
    # It used to validate parameter within the `_fit_context` decorator.
    _parameter_constraints = {
        "demo_param": [str],
    }

    def __init__(self, demo_param="demo"):
        self.demo_param = demo_param

    @_fit_context(prefer_skip_nested_validation=True)
    def fit(self, X, y=None):
        """A reference implementation of a fitting function for a transformer.
        一般就是记录训练集信息，比如统计量mean，std
        Parameters
        ----------
        X : {array-like, sparse matrix}, shape (n_samples, n_features)
            The training input samples.

        y : None
            There is no need of a target in a transformer, yet the pipeline API
            requires this parameter.

        Returns
        -------
        self : object
            Returns self.
        """
        X = self._validate_data(X, accept_sparse=True)

        # Return the transformer
        return self

    def transform(self, X):
        """A reference implementation of a transform function.

        Parameters
        ----------
        X : {array-like, sparse-matrix}, shape (n_samples, n_features)
            The input samples.

        Returns
        -------
        X_transformed : array, shape (n_samples, n_features)
            The array containing the element-wise square roots of the values
            in ``X``.
        """
        # Since this is a stateless transformer, we should not call `check_is_fitted`.
        # Common test will check for this particularly.

        # Input validation
        # We need to set reset=False because we don't want to overwrite `n_features_in_`
        # `feature_names_in_` but only check that the shape is consistent.
        X = self._validate_data(X, accept_sparse=True, reset=False)
        return np.sqrt(X)

    def _more_tags(self):
        # This is a quick example to show the tags API:\
        # https://scikit-learn.org/dev/developers/develop.html#estimator-tags
        # Here, our transformer does not do any operation in `fit` and only validate
        # the parameters. Thus, it is stateless.
        return {"stateless": True}

默认地，fit不允许X中有nan，这通过tags实现